## tl;dr
B10/B11 第一处分界预设 80%，其余参数重新拟合最新真实数据。历史均值方案单独保存，未混入主候选。

## Context & Methods
### Key Assumptions
95%→20%；每段至少10个百分点/20秒；80%是明确约束，不是自由搜索结果。旧条件未确认相同。
这个核查本是模型的 companion。执行方式：所有普通 Python 单元逐格运行；未验证 Jupyter 内核执行。

In [1]:
from pathlib import Path
import json, hashlib, sys
import pandas as pd
root=Path('/Users/alexmason/Downloads/drone_experiment')
sys.path.insert(0,str(root))
from output_py.revise_b10_b11_calibration import review_sources, pooled_curve
from output_py.search_bideal_boundaries import fit
folder=Path('/Users/alexmason/Downloads/drone_experiment/analysis_results/battery_normalization_revision_20260909')
model=json.loads((folder/'model.json').read_text())
traces, profiles, individual=review_sources()
print('Reviewed hover records:',len(profiles))

Reviewed hover records: 11


## Data
实时重读原始记录及 SHA256；只把当前同一电池/无人机配对用于新旧平均对照。

In [2]:
print(pd.DataFrame(profiles)[['battery_id','drone_id','run_id','use']].to_string(index=False))

battery_id drone_id                         run_id                                   use
       B10  drone_1                20260830_174908                     excluded_no_hover
       B10  drone_1                20260830_174922             excluded_incomplete_95_20
       B11  drone_1                20260513_143201 primary_current_or_pooled_sensitivity
       B11  drone_1                20260514_135933                     excluded_no_hover
       B11  drone_1                20260514_151213                     excluded_no_hover
       B11  drone_1                20260906_164629                     excluded_no_hover
       B11  drone_1                20260906_164725 primary_current_or_pooled_sensitivity
       B10  drone_2                20260513_180556 primary_current_or_pooled_sensitivity
       B10  drone_2                20260906_172945                     excluded_no_hover
       B10  drone_2                20260906_173013 primary_current_or_pooled_sensitivity
       B10  drone_3 b

## Results
重新计算约束模型，并核对保存的两个电池系数；不覆盖模型。

In [3]:
for battery in ['B10','B11']:
    item=model['batteries'][battery]
    trace=next(t for t in traces if t['battery_id']==battery and t['run_id']==item['run_id'])
    result=fit(trace,80,int(item['boundaries_soc'][2]),20)
    assert abs(result['rmse']-item['in_sample_rmse_pp'])<1e-10
    print(battery,item['boundaries_soc'],'RMSE',result['rmse'])

B10 [95.0, 80.0, 56.0, 20.0] RMSE 1.2678976971902565
B11 [95.0, 80.0, 56.0, 20.0] RMSE 1.469413822548195


In [4]:
for battery,drone in [('B10','drone_2'),('B11','drone_1')]:
    selected=[t for t in traces if t['battery_id']==battery and t['drone']==drone]
    curve, _, grid, _, _=pooled_curve(selected)
    print('Old/new mean sensitivity:',battery,curve.boundaries,'approximation RMSE',float(grid.iloc[0].rmse_pp))

Old/new mean sensitivity: B10 (95.0, 85.0, 56.0, 20.0) approximation RMSE 1.1534262304676997
Old/new mean sensitivity: B11 (95.0, 82.0, 51.0, 20.0) approximation RMSE 1.0142768448729433


## Takeaways
主候选修改了分界假设，样本内误差略增；历史资料支持约80%附近的分界，但新旧高电量耗时差异大，不证明可直接混合。原始数据、上一版及飞行逻辑未改。